# 06 · Unity Catalog: gobernar lo que construimos

⏱ **14 min** · Slides: 24–32 (gobierno, jerarquía, objetos, tablas administradas vs externas, permisos, linaje, pasos)

**Objetivo:** ver el pipeline desde el gobierno: dónde vive cada objeto, quién puede verlo y de dónde viene.

| Paso | Qué haces | Slide |
|---|---|---|
| 1 | La jerarquía: catálogo → esquema → objetos | 26 |
| 2 | Inventario con `information_schema` | 27 |
| 3 | Tablas administradas vs externas | 28 |
| 4 | Una vista que oculta columnas | 27 |
| 5 | Permisos: `GRANT`, `SHOW GRANTS`, `REVOKE` | 30 |
| 6 | Linaje y auditoría | 31 |
| 7 | Azure vs Free Edition | 29, 32 |

**Requiere:** notebook 04.

In [0]:
%run ./00_Setup

## Paso 1 · La jerarquía

In [0]:
%sql
SELECT current_catalog() AS catalogo, current_schema() AS esquema, current_user() AS usuario

In [0]:
%sql
SHOW TABLES

In [0]:
%sql
SHOW VOLUMES

In [0]:
# Nombre de tres niveles: catálogo.esquema.tabla
print(qualified_name(SILVER))
print("Filas:", spark.table(qualified_name(SILVER)).count())

👀 **Qué observar:** en la Free Edition el catálogo es `workspace`; en Azure sería, por ejemplo, `prod`. El esquema y las tablas son los nuestros.

## Paso 2 · Inventario con `information_schema`

Unity Catalog expone sus metadatos como tablas consultables.

In [0]:
display(spark.sql(f"""
    SELECT table_name, table_type, data_source_format, created
    FROM `{CATALOG}`.information_schema.tables
    WHERE table_schema = '{SCHEMA}'
    ORDER BY table_name
"""))

In [0]:
display(spark.sql(f"""
    SELECT column_name, data_type
    FROM `{CATALOG}`.information_schema.columns
    WHERE table_schema = '{SCHEMA}' AND table_name = '{SILVER}'
    ORDER BY ordinal_position
"""))

## Paso 3 · Tablas administradas vs externas

In [0]:
%sql
DESCRIBE EXTENDED silver_episodes

👀 **Qué observar:** `Type: MANAGED` y una `Location` que administra Databricks. Una tabla **externa** tendría una ruta que tú
eliges dentro de una *external location*; en la Free Edition no hay almacenamiento propio, así que todas son administradas.
Lo mismo con el volumen:

In [0]:
%sql
DESCRIBE VOLUME datos

## Paso 4 · Una vista que oculta columnas

Los analistas no necesitan `source_file` ni el id de la serie.

In [0]:
%sql
CREATE OR REPLACE VIEW silver_episodes_public AS
SELECT episode_id, episode_name, airdate, show_name, network_name, episode_rating
FROM silver_episodes

In [0]:
%sql
SELECT * FROM silver_episodes_public LIMIT 5

## Paso 5 · Permisos: `GRANT`, `SHOW GRANTS`, `REVOKE`

Se dan con SQL estándar y se heredan hacia abajo. `account users` es el grupo con todos los usuarios de la cuenta
(en la Free Edition, solo tú). En Azure usarías grupos como `analistas`.

In [0]:
%sql
GRANT USE SCHEMA ON SCHEMA lab_tvmaze_m34 TO `account users`

In [0]:
%sql
-- las vistas se gobiernan igual que las tablas
GRANT SELECT ON TABLE silver_episodes_public TO `account users`

In [0]:
%sql
SHOW GRANTS ON TABLE silver_episodes_public

In [0]:
%sql
REVOKE SELECT ON TABLE silver_episodes_public FROM `account users`

👀 **Qué observar:** para leer la vista hace falta la cadena completa: `USE CATALOG` + `USE SCHEMA` + `SELECT`.
`SHOW GRANTS` muestra quién tiene qué, y `REVOKE` lo quita. Todo queda en un solo lugar para todos los workspaces.

## Paso 6 · Linaje y auditoría

**Linaje (en la interfaz):** Catalog → `lab_tvmaze_m34` → `gold_network_daily` → pestaña **Lineage**.
Verás `bronze_schedule → silver_episodes → gold_network_daily` y los notebooks que escribieron cada una, sin configurar nada.

**Auditoría (tablas del sistema):** `system.access.table_lineage` y `system.access.audit`. Puede que no estén habilitadas en tu cuenta:

In [0]:
try:
    display(spark.sql(f"""
        SELECT source_table_name, target_table_name, entity_type, event_time
        FROM system.access.table_lineage
        WHERE target_table_schema = '{SCHEMA}'
        ORDER BY event_time DESC
        LIMIT 10
    """))
except Exception as e:
    print("Las tablas del sistema no están disponibles en este workspace:", str(e).splitlines()[0][:120])
    print("Usa la pestaña Lineage de Catalog Explorer: muestra el mismo recorrido.")

## Paso 7 · Azure vs Free Edition

| Slide (Azure) | Free Edition | En este laboratorio |
|---|---|---|
| Access connector + storage credential | No se configuran (no hay almacenamiento propio) | — |
| External location | No disponible | Todo es administrado (tablas y volumen) |
| Metastore por región | Ya creado y asignado | `SHOW CATALOGS` |
| Catálogo por entorno (`prod`, `dev`) | Un catálogo: `workspace` | `current_catalog()` |
| Grupos (`analistas`, `ingenieria`) | Grupo `account users` | `GRANT ... TO account users` |
| Linaje y auditoría | Linaje en Catalog Explorer; tablas del sistema según la cuenta | Paso 6 |

## ✔ Checkpoint

- [ ] Ubico mis tablas con nombre de tres niveles y en `information_schema`
- [ ] Sé por qué aquí todo es administrado y qué cambiaría con una *external location*
- [ ] Creé una vista, di un permiso, lo vi con `SHOW GRANTS` y lo quité
- [ ] Encontré el linaje bronze → silver → gold

**Siguiente:** `07_Pipeline_Completo_y_Cierre`